# Sprint 1 · Trace, extend and explain one pipeline

Linden Facilities needs 24 hourly energy-demand estimates in one CSV within 60 seconds. You will trace its five stages, follow one validation edit, review an unsafe proposal and justify how predictions should be delivered.

**Use this optional notebook to explore the sprint.** Follow the six sections below in order. Each Campus page supplies the explanation or brief for that section; run its notebook cells once and keep answers here. A later checkpoint reuses earlier work. Running all reference cells does not complete the independent tasks.

| Campus activity | Notebook section | Keep, rather than redo |
|---|---|---|
| [Pipeline purpose](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/01-from-notebook-to-pipeline/lesson.md) | 1 · Understand the five stages | Your stage map |
| [Trace the scaffold](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/02-reading-the-provided-pipeline-scaffold/lesson.md) | 2 · Follow raw input to predictions | One row trace and baseline batch |
| [Follow the expert edit](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/03-adding-and-validating-a-bounded-pipeline-stage/lesson.md) | 3 · Add the validation stage | One edited project and before/after checks |
| [Review generated code](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/04-explain-the-code-checkpoint/lesson.md) | 4 · Review a proposed guard | Explanation and distinguishing probes |
| [Choose an inference pattern](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/05-batch-vs-real-time-inference-tradeoffs/lesson.md) | 5 · Decide how to deliver | Recommendation using the existing batch evidence |
| [Client checkpoint](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/06-pipeline-extension-deliverable/lesson.md) | 6 · Apply the contract to Harbor | Supplier decisions plus reused sprint work |

**Start locally:** follow [setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select the project's Python 3.12 environment and run the provided setup cell once. Its package checks are preparation, not code you must explain. Work is retained under the printed local `work` folder. Setup resumes the saved project record; it does not create another checkout or tracking store.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use Python 3.12. Locally run uv sync --locked --python 3.12 and select .venv/bin/python.')
# Hosted exploration is optional. Local setup uses the already installed editable project.
try:
    import linden_mlops
except ImportError:
    if 'google.colab' not in sys.modules:
        raise RuntimeError('Follow docs/setup.md from the repository root, then select its .venv kernel.')
    hosted_root = Path('/content/ml-llm-ops')
    if not hosted_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/richhiey/ml-llm-ops.git', str(hosted_root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(hosted_root), 'pytest==8.4.2'], check=True)
    sys.path.insert(0, str(hosted_root / 'src'))
    import linden_mlops
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate, run_pipeline,
                          export_runs, verify_summary, register, approve, activate, batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
print('Working source:', ROOT)
print('Retained work:', state)
print('Code:', code_revision())
print('Saved decisions:', sorted(progress.data))

## 1 · Understand the five stages

A notebook is a useful interface, but an in-memory result can depend on cells run earlier. A **pipeline** makes the sequence and each stage's input/output explicit. **Reproducible** means another person can recover the same recipe and obtain an agreed result; it does not mean executing a cell twice.

| Stage | Input → output |
|---|---|
| Ingestion | Historical CSV → checked table |
| Preprocessing | Checked table → ordered model features |
| Training | Training features and known demand → fitted scaler and model |
| Evaluation | Held-out features and known demand → error measurements |
| Inference | Planned features and fitted model → predictions |

Here a **feature** is a model input; `demand_kwh` is the known training **target**. A fitted scaler stores means and scales learned from training rows. Keep it with the model: tomorrow's rows must use those stored values, not fit a new scaler.

**Pause:** sketch the five-stage map here. Mark where known demand is needed and where a planned row can pass without it. The next section gives concrete objects for this same map, not another report.

## 2 · Follow raw input to predictions

**Read the input.** `ingest()` returns a pandas DataFrame, a table with named columns. `row_id` is the unique identifier. `partition` states each historical row's role: 168 training, 48 validation and 24 reserved final rows. The final rows stay out of this sprint's decisions.

Run the next cell and choose one historical row to trace. Expect the three counts and the columns `temperature_c`, `hour`, `occupancy`, `demand_kwh` and `partition`.

In [ ]:
import inspect, json, contextlib, io, runpy, shutil
import pandas as pd
import numpy as np
from linden_mlops.pipeline import DATA, FEATURES, TARGET
frame = ingest()
print(frame.groupby('partition').size())
display(frame.head())

### Select features, fit once, evaluate held-out rows

`preprocess` selects the three ordered features; it does not fit statistics. A Boolean **mask** is one true/false value per row; `.loc[mask]` selects those rows. The same mask must select matching features and targets.

`train` creates a fitted `StandardScaler` and Ridge regressor together. Ridge is a linear prediction model; `alpha=1` is its supplied configuration, not a value to tune here. `evaluate` predicts validation rows without fitting again. **RMSE** is root mean squared error, in kWh; **MAE** is mean absolute error. Lower is better. Expect baseline validation RMSE about `1.443` kWh; this is a reproducible teaching result, not a claim about real buildings.

In [ ]:
config = Config()
features = preprocess(frame, config)
train_rows = frame.partition == 'train'
valid_rows = frame.partition == 'validation'
model = train(features.loc[train_rows], frame.loc[train_rows, TARGET], config)
metrics = evaluate(model, features.loc[valid_rows], frame.loc[valid_rows, TARGET])
print('Feature order:', features.columns.tolist())
print('Validation:', metrics)
assert not (train_rows & valid_rows).any()
assert features.index.equals(frame.index)
assert abs(metrics['rmse_kwh'] - 1.4430039250255964) < 1e-6

### Find the learned state

Predict the number of fitting rows, validation rows and model inputs before running. Expect `168`, `48`, steps named `scale` and `ridge`, and `3` features. An attribute ending in an underscore, such as `n_features_in_`, records fitted state; `alpha` is a configuration setting.

In [ ]:
print('Train / validation:', int(train_rows.sum()), int(valid_rows.sum()))
print('Fitted steps:', list(model.named_steps))
print('Input features:', model.named_steps['scale'].n_features_in_)

### Follow planned input to the delivered file

Planned rows have no known `demand_kwh`. The supplied `bootstrap` wrapper runs the same recipe from saved code and makes a fitted model available for `batch_predict`. Model selection/storage is prepared infrastructure here; you do not need to configure a registry or explain its internals in this sprint.

Expect a file with 24 finite `prediction_kwh` values, preserving planned `row_id` order. The returned metadata identifies the source execution and model version. Compare its validation metric with the trace above: the delivered model comes from that same baseline recipe. Timing measures this complete batch's model load, prediction and output write.

In [ ]:
baseline_state = state / 'baseline'
if not (baseline_state / 'result.json').exists():
    with contextlib.redirect_stdout(io.StringIO()):
        delivery = course_cli('bootstrap', '--state', str(baseline_state))
    (baseline_state / 'result.json').write_text(delivery.stdout)
baseline = json.loads((baseline_state / 'result.json').read_text())
predictions = pd.read_csv(baseline_state / 'batch.csv')
planned = pd.read_csv(DATA / 'batch-v1.csv')
evidence = baseline['inference']
display(planned.head())
display(predictions.head())
print('Delivered file:', baseline_state / 'batch.csv')
print('Batch:', evidence)
assert planned.row_id.tolist() == predictions.row_id.tolist()
assert np.isfinite(predictions.prediction_kwh).all() and len(predictions) == 24
assert baseline['training']['metrics'] == metrics
assert evidence['complete_within_60s']

**Finish the same stage map:** add one historical `row_id`, its partition and target role, then one planned `row_id` and its output prediction. Which object owns fitted scaling? Why can't a completed prediction file measure tomorrow's prediction error? Keep these short answers here and continue; do not rerun the baseline for later pages.

## 3 · Add the validation stage

Linden receives a numeric `hour=24`. A finite-number check alone accepts it. The new **contract** rejects temperature outside `[-30,50]`, hour outside integer `[0,23]`, and occupancy outside `[0,100]`. Accepted input must return unchanged in a separate returned table; invalid input must raise `ValueError`, not be silently repaired.

Use the current repository as your editable project. The next cell saves the reference function in `work/` and temporarily installs a finite-only starting function in `src/linden_mlops/pipeline.py`. This controlled exercise changes one function; do not run it over unrelated guard edits. The existing focused test checks the proposed contract. Its first failure is intentional: expect `DID NOT RAISE`. A good validation score on ordinary rows cannot expose this missing rule.

In [ ]:
target = ROOT / 'src/linden_mlops/pipeline.py'
reference_file = state / 'validation-reference.txt'
if not progress.data.get('stage_complete'):
    source = target.read_text()
    begin = source.index('def validate_ranges(')
    end = source.index(chr(10) * 2 + 'def preprocess(', begin)
    starter = 'def validate_ranges(frame):' + chr(10) + '    return validate_features(frame).copy(deep=True)'
    if not reference_file.exists():
        reference_file.write_text(source[begin:end])
    elif source[begin:end].strip() not in {starter.strip(), reference_file.read_text().strip()}:
        raise ValueError('Your guard has other edits; preserve them before starting the controlled exercise')
    target.write_text(source[:begin] + starter + source[end:])
    _ = project_tests(ROOT, 'tests/test_spine.py::test_contract_extension_and_unchanged_pipeline',
                      '-q', expected_returncode=1)
else:
    print('Reusing the saved completed stage; no source reset or repeated test required.')
print('Working source:', target)

### Follow the expert's bounded edit

The function below is the expert reference. Read the code alongside this explanation before applying it:

| Statement | Why it is needed |
|---|---|
| `validate_features(frame)` | Reject empty input, missing feature columns and non-finite numbers first |
| `for column, low, high ...` | Apply a declared rule to each feature |
| `.between(low, high).all()` | Require every row to lie within inclusive bounds |
| `raise ValueError(...)` | Stop at invalid input instead of hiding it |
| `frame.hour % 1 == 0` | Reject fractional hours that pass the range check |
| `return frame.copy(deep=True)` | Return only after every check; preserve caller-owned input |

The next cell prints the saved reference function and applies only this function to your working source, showing the real source diff. This is a follow-the-expert activity; you do not need to type a second implementation. `preprocess` already calls this boundary when `range_guard=True`, so ingestion, fitting and evaluation stay intact.

In [ ]:
import difflib
expert = reference_file.read_text()
print(expert)
before = target.read_text()
if not progress.data.get('stage_complete'):
    begin = before.index('def validate_ranges(')
    end = before.index(chr(10) * 2 + 'def preprocess(', begin)
    after = before[:begin] + expert + before[end:]
    target.write_text(after)
    stage_diff = ''.join(difflib.unified_diff(before.splitlines(True), after.splitlines(True),
                                            fromfile='starter', tofile='src/linden_mlops/pipeline.py'))
    _ = (state / 'stage-change.diff').write_text(stage_diff)
print((state / 'stage-change.diff').read_text())

### Check the contract, including caller-owned input

`runpy.run_path` reads your saved source file into a separate namespace. `project_guard` below is therefore your edited function, not the notebook's earlier reference import. If you edit that file again, rerun this loading cell before probing it.

Predict `hour=0`, `23`, `1.5` and `24` before execution. Expect the first two accepted and the last two rejected. `assert_frame_equal` checks values, columns, index and types; `is not` checks that the returned object is a distinct copy. A `try/except` block lets us record an expected rejection without stopping the notebook.

In [ ]:
project_guard = runpy.run_path(str(target))['validate_ranges']
original = frame.copy(deep=True)
checked = project_guard(frame)
pd.testing.assert_frame_equal(frame, original)
pd.testing.assert_frame_equal(checked, original)
assert checked is not frame
observed = []
for hour in [0, 23, 1.5, 24]:
    sample = frame.iloc[[0]].copy()
    sample['hour'] = float(hour)
    try:
        project_guard(sample)
        status = 'accepted'
    except ValueError:
        status = 'rejected'
    observed.append({'hour': hour, 'observed': status})
display(pd.DataFrame(observed))
assert [r['observed'] for r in observed] == ['accepted', 'accepted', 'rejected', 'rejected']

### Confirm the unchanged stages still run

Rerun the same focused test: expect one passing test. It also checks temperature/occupancy boundaries, missing and non-finite features, and unchanged valid-data metrics. `project_tests` and `course_cli(..., project=project)` use fresh processes against your saved source; this avoids stale notebook imports.

Run the edited project's full delivery path once. Expect the same validation metrics, all 24 output rows and completion within 60 seconds. Keep this result for the inference decision and checkpoint. Registry lifecycle tests and fresh-environment recovery are later work, not extra Sprint 1 assignments.

In [ ]:
if not progress.data.get('stage_complete'):
    project_tests(ROOT, 'tests/test_spine.py::test_contract_extension_and_unchanged_pipeline', '-q')
    with contextlib.redirect_stdout(io.StringIO()):
        delivery = course_cli('bootstrap', '--state', str(state / 'edited'), project=ROOT)
    completed = json.loads(delivery.stdout)
    (state / 'stage-result.json').write_text(json.dumps(completed, indent=2))
    progress.save(stage_complete={'code_revision': code_revision(),
                                 'result_file': str(state / 'stage-result.json')})
completed = json.loads(Path(progress.require('stage_complete')['result_file']).read_text())
if completed['training']['code_revision'] != code_revision():
    raise ValueError('Source changed since this stage result; verify the affected change before continuing')
edited_predictions = pd.read_csv(state / 'edited/batch.csv')
assert completed['training']['metrics'] == metrics
assert planned.row_id.tolist() == edited_predictions.row_id.tolist()
assert np.isfinite(edited_predictions.prediction_kwh).all()
assert completed['inference']['rows'] == 24 and completed['inference']['complete_within_60s']
print('Edited-project batch:', completed['inference'])
print('Outputs:', state / 'edited')

## 4 · Review a proposed guard

An assistant proposes the function below. Treat it as untrusted exercise material, not an update to install. Your project already contains the explained guard: keep it. **Mutation** means changing the object passed by the caller.

1. Explain each candidate statement and predict what it does to `hour=24`, a negative hour and a fractional hour.
2. Probe those cases plus one valid input. Compare the caller's values before/after and whether the returned object is the same object. Keep at least one observed failure against the required contract.
3. Explain the changes needed before merging. Use your existing `project_guard` to contrast correct behavior; do not rebuild the project or rerun every pipeline stage.

The Campus hints and pandas documentation are available if needed. The broader boundary and integration checks above remain evidence; repeat only a check affected by a new edit.

In [ ]:
candidate_path = ROOT / 'docs/course/exercises/generated-stage-candidate.py'
print(candidate_path.read_text())
proposed_guard = runpy.run_path(str(candidate_path))['proposed_guard']

In [ ]:
# Add your predictions and probes here.
# Use a fresh sample.copy(deep=True) for each case.
# Keep the unsafe candidate out of target (your working pipeline source).

## 5 · Decide how to deliver predictions

Use the Campus comparison to decide from the brief: all 24 inputs arrive together; one complete file is needed within 60 seconds; there is no interactive caller.

**Latency** is time until the required result is ready. **Throughput** is records processed per unit time. A batch can load one model for the whole file and exit. A request service must respond as inputs arrive and meet an availability window. Actual cost depends on workload and resources; no endpoint benchmark or price is supplied.

Add one short recommendation here addressing latency, throughput and resource obligations. Cite `completed['inference']` from your edited project and explain what different requirement would make request serving appropriate. Do not run another batch or build an endpoint.

## 6 · Apply the contract to Harbor

Harbor Facilities supplies a new five-row intake below. It has only `row_id`, `temperature_c`, `hour` and `occupancy`: no historical target or partition. Apply the taught feature-validation contract; do not pass this file to historical `ingest()`, train on it or repair its values.

Read the Campus client brief. Classify each identifier, justify its rule, then test your decisions using the already loaded `project_guard`. Preserve the raw table. Decide whether the mixed intake can pass as one whole batch. This is new input for the same project, not a new pipeline build.

Keep your work together in this repository and its work folder: your source diff and explanation, candidate review, supplier decisions, passing contract/batch evidence and inference recommendation. Reuse previous outputs; no new notebook, extra report, registry setup or clean-environment run is required.

In [ ]:
supplier_path = ROOT / 'docs/course/exercises/harbor-intake.csv'
supplier = pd.read_csv(supplier_path)
supplier_original = supplier.copy(deep=True)
display(supplier)

In [ ]:
# Add your row classifications, reasons and guard probes here.
# Check the mixed batch as well as individual rows.
# Confirm supplier still equals supplier_original after your checks.

## Finish and keep your work

Save your answers in this notebook. Keep the printed local work folder: it contains the source diff, progress record and run outputs. The editable project is this repository’s `src/` tree. Your edited project's batch CSV and metadata are under `work/edited/`. Keep this clone and work folder for later sprints. Sprints 2–4 reopen the same progress record and selected run. Use the recovery export when handing artifacts to another computer.

Do not run setup again simply to finish a Campus page. Restarting the kernel loses Python variables, but reopening this notebook resumes the same local work. Completed source edits and model runs are reused. Use an explicit new `LINDEN_WORK_DIR` only for a deliberate separate experiment.